# DS4/GSE278450 — memory-safe post-QC Scrublet diagnostic

This notebook performs the DS4 doublet diagnostic on the complete post-QC
dataset without loading both `X` and `layers["counts"]` into memory.

The historical diagnostic reported:

- 310 predicted doublets;
- 87,784 predicted singlets.

The output remains diagnostic only: predicted doublets are retained and are
not removed from the main analysis.

Runtime compatibility changes:

- read only `layers["counts"]` directly from the H5AD CSR storage;
- reproduce Scanpy's pre-filtering and default HVG selection before the
  Scrublet core;
- use a deterministic 64-bit-safe simulated-pair sampler;
- use `mean_center=False`, so Scrublet uses sparse TruncatedSVD rather than
  the dense centered-PCA path that exceeds the available RAM;
- copy the input H5AD on disk and patch only `obs` and `uns`.

Because centered PCA is replaced by sparse TruncatedSVD, exact cell-level
scores and the historical count of 310 are checked and reported, but are not
asserted.


In [1]:
import gc
import importlib.util
import inspect
import json
import math
import os
import shutil
import sys
from contextlib import contextmanager
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import anndata as ad
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from sklearn.utils import check_random_state
from sklearn.utils.sparsefuncs import inplace_csr_row_scale

try:
    from anndata.io import read_elem
except ImportError:
    # Compatibility with older AnnData releases.
    from anndata._io.specs import read_elem

print("Python:", sys.version)
print("anndata:", ad.__version__)
print("scanpy:", sc.__version__)


Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
anndata: 0.12.7
scanpy: 1.11.5


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_33448\528027515.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)
C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_33448\528027515.py:32: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


In [2]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
        ):
            return candidate

    return start


PROJECT_ROOT = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(Path.cwd()),
    )
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE278450_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS4_GSE278450"
        / "GSE278450_postQC.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE278450_SCRUBLET_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS4_GSE278450"
        / "GSE278450_postQC_scrublet_checked.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT
    / "results"
    / "qc"
    / "DS4_GSE278450"
    / "00c_doublet_diagnostic"
)
FIGURE_DIR = AUDIT_DIR / "figures"

EXPECTED_INPUT_SHAPE = (88_094, 24_584)
EXPECTED_PREDICTED_DOUBLETS = 310
EXPECTED_PREDICTED_SINGLETS = 87_784

EXPECTED_DOUBLET_RATE = 0.05
SIM_DOUBLET_RATIO = 2.0
N_PCS = 30
RANDOM_SEED = 0

SCRUBLET_MIN_CELLS_PER_GENE = 3
SCRUBLET_MIN_GENES_PER_CELL = 3
GENE_NNZ_CHUNK = int(
    os.environ.get(
        "GSE278450_GENE_NNZ_CHUNK",
        "1000000",
    )
)

RUNTIME_PATCH_VERSION = (
    "ds4-global-direct-hdf5-prehvg-int64-svd-v2"
)

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Post-QC input not found: {INPUT_H5AD}\n"
        "Run 00b_ds4_quality_control.ipynb first or set "
        "GSE278450_POSTQC_H5AD."
    )

if importlib.util.find_spec("scrublet") is None:
    raise ImportError(
        "The 'scrublet' package is required by scanpy.pp.scrublet."
    )

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)
print("Runtime patch:", RUNTIME_PATCH_VERSION)


Input: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS4_GSE278450\GSE278450_postQC.h5ad
Output: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\DS4_GSE278450\GSE278450_postQC_scrublet_checked.h5ad
Runtime patch: ds4-global-direct-hdf5-prehvg-int64-svd-v2


In [3]:
def decode_h5_attr(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")

    return value


def stored_matrix_shape(node) -> tuple[int, int]:
    if isinstance(node, h5py.Dataset):
        return tuple(int(value) for value in node.shape)

    if not isinstance(node, h5py.Group):
        raise TypeError(
            "Unsupported HDF5 matrix node: "
            f"{type(node).__name__}"
        )

    shape = node.attrs.get("shape")
    if shape is None:
        raise KeyError(
            f"Sparse matrix group {node.name!r} has no shape."
        )

    return tuple(int(value) for value in shape)


def stored_sparse_encoding(group: h5py.Group) -> str:
    return str(
        decode_h5_attr(
            group.attrs.get("encoding-type", "")
        )
    )


def read_h5_dataset_as(
    dataset: h5py.Dataset,
    dtype,
) -> np.ndarray:
    output = np.empty(dataset.shape, dtype=dtype)
    dataset.read_direct(output)
    return output


def load_full_csr(group: h5py.Group) -> sp.csr_matrix:
    encoding = stored_sparse_encoding(group)

    if encoding != "csr_matrix":
        raise RuntimeError(
            "The memory-safe DS4 reader requires CSR storage; "
            f"observed {encoding!r}."
        )

    shape = stored_matrix_shape(group)

    print(
        "Reading counts.data:",
        f"{group['data'].shape[0]:,} entries",
    )
    data = read_h5_dataset_as(
        group["data"],
        np.float32,
    )

    print("Reading counts.indices")
    indices = read_h5_dataset_as(
        group["indices"],
        np.int32,
    )

    print("Reading counts.indptr")
    indptr = read_h5_dataset_as(
        group["indptr"],
        np.int64,
    )

    matrix = sp.csr_matrix(
        (data, indices, indptr),
        shape=shape,
        dtype=np.float32,
    )

    if matrix.nnz != len(data):
        raise AssertionError(
            "The reconstructed CSR matrix has an unexpected "
            "nonzero count."
        )

    return matrix


def iter_array_chunks(array, chunk_size: int):
    for start in range(0, len(array), chunk_size):
        yield array[start : start + chunk_size]


def assert_raw_counts_chunked(
    matrix: sp.csr_matrix,
    label: str,
    chunk_size: int = 1_000_000,
):
    if not sp.isspmatrix_csr(matrix):
        raise TypeError(
            f"{label} must be CSR; observed "
            f"{type(matrix).__name__}."
        )

    if matrix.nnz == 0:
        raise ValueError(f"{label} contains no nonzero values.")

    for data_chunk in iter_array_chunks(
        matrix.data,
        chunk_size,
    ):
        if not np.isfinite(data_chunk).all():
            raise ValueError(
                f"{label} contains non-finite values."
            )

        if np.any(data_chunk < 0):
            raise ValueError(
                f"{label} contains negative values."
            )

        rounded = np.rint(data_chunk)
        if not np.array_equal(data_chunk, rounded):
            raise ValueError(
                f"{label} is not an integer raw-count matrix."
            )


def column_nnz_chunked(
    csr: sp.csr_matrix,
    chunk_size: int,
) -> np.ndarray:
    result = np.zeros(
        csr.shape[1],
        dtype=np.int64,
    )

    for index_chunk in iter_array_chunks(
        csr.indices,
        chunk_size,
    ):
        result += np.bincount(
            index_chunk,
            minlength=csr.shape[1],
        )

    return result


def select_scanpy_default_hvgs(
    raw_counts: sp.csr_matrix,
    raw_var_names: pd.Index,
):
    row_nnz = np.diff(raw_counts.indptr)
    cell_keep = (
        row_nnz >= SCRUBLET_MIN_GENES_PER_CELL
    )

    if not cell_keep.all():
        working_raw = raw_counts[cell_keep, :].copy()
    else:
        working_raw = raw_counts

    gene_nnz = column_nnz_chunked(
        working_raw,
        chunk_size=GENE_NNZ_CHUNK,
    )
    gene_keep = (
        gene_nnz >= SCRUBLET_MIN_CELLS_PER_GENE
    )

    n_genes_after_min_cells = int(
        gene_keep.sum()
    )

    if n_genes_after_min_cells <= N_PCS:
        raise RuntimeError(
            "Too few genes remain after min_cells=3 "
            f"filtering: {n_genes_after_min_cells}."
        )

    filtered_var_names = raw_var_names[gene_keep]

    print(
        "Creating the one temporary normalized matrix for "
        "default HVG selection"
    )
    normalized_log = working_raw[:, gene_keep].copy()

    cell_totals = np.asarray(
        normalized_log.sum(axis=1)
    ).ravel()

    positive_totals = cell_totals[
        cell_totals > 0
    ]
    if positive_totals.size == 0:
        raise ValueError(
            "All filtered cells have zero total counts."
        )

    target_sum = float(
        np.median(positive_totals)
    )

    scale = np.divide(
        target_sum,
        cell_totals,
        out=np.zeros_like(
            cell_totals,
            dtype=np.float32,
        ),
        where=cell_totals > 0,
    )

    inplace_csr_row_scale(
        normalized_log,
        scale.astype(
            np.float32,
            copy=False,
        ),
    )
    np.log1p(
        normalized_log.data,
        out=normalized_log.data,
    )

    hvg_adata = ad.AnnData(
        X=normalized_log,
        var=pd.DataFrame(
            index=filtered_var_names.copy()
        ),
    )

    sc.pp.highly_variable_genes(hvg_adata)

    if "highly_variable" not in hvg_adata.var:
        raise RuntimeError(
            "HVG selection did not create a mask."
        )

    hvg_within_filtered = (
        hvg_adata.var["highly_variable"]
        .to_numpy(dtype=bool)
    )
    n_hvg = int(
        hvg_within_filtered.sum()
    )

    del hvg_adata
    del normalized_log
    gc.collect()

    if n_hvg <= N_PCS:
        raise RuntimeError(
            f"Only {n_hvg} highly variable genes were "
            f"selected; more than {N_PCS} are required."
        )

    full_hvg_mask = np.zeros(
        raw_counts.shape[1],
        dtype=bool,
    )
    full_hvg_mask[
        np.flatnonzero(gene_keep)[
            hvg_within_filtered
        ]
    ] = True

    hvg_counts = (
        working_raw[:, full_hvg_mask]
        .copy()
        .tocsr()
    )
    hvg_var_names = raw_var_names[
        full_hvg_mask
    ]

    if working_raw is not raw_counts:
        del working_raw

    gc.collect()

    return {
        "cell_keep": cell_keep,
        "hvg_counts": hvg_counts,
        "hvg_var_names": hvg_var_names,
        "genes_after_min_cells": (
            n_genes_after_min_cells
        ),
        "highly_variable_genes": n_hvg,
        "normalization_target_sum": target_sum,
    }


import scanpy.preprocessing._utils as _scanpy_pp_utils
import scanpy.preprocessing._scrublet.core as _scrublet_core


_ORIGINAL_UTILS_SAMPLE_COMB = (
    _scanpy_pp_utils.sample_comb
)
_ORIGINAL_CORE_SAMPLE_COMB = (
    _scrublet_core.sample_comb
)
_INT32_MAX = int(np.iinfo(np.int32).max)


def sample_comb_int64_safe(
    dims,
    nsamp,
    random_state=None,
    method="auto",
):
    dims = tuple(int(value) for value in dims)
    nsamp = int(nsamp)
    population = int(math.prod(dims))

    if any(value <= 0 for value in dims):
        raise ValueError(
            "All sampling dimensions must be positive; "
            f"observed {dims}."
        )

    if nsamp < 0 or nsamp > population:
        raise ValueError(
            f"Cannot sample {nsamp} unique combinations "
            f"from a population of {population}."
        )

    if population <= _INT32_MAX:
        return _ORIGINAL_UTILS_SAMPLE_COMB(
            dims,
            nsamp,
            random_state=random_state,
            method=method,
        )

    rng = check_random_state(random_state)
    selected_flat = np.empty(
        nsamp,
        dtype=np.int64,
    )
    selected_set = set()

    position = 0
    while position < nsamp:
        candidate = int(
            rng.randint(
                0,
                population,
                dtype=np.int64,
            )
        )

        if candidate in selected_set:
            continue

        selected_set.add(candidate)
        selected_flat[position] = candidate
        position += 1

    return np.column_stack(
        np.unravel_index(
            selected_flat,
            dims,
        )
    ).astype(np.int64, copy=False)


@contextmanager
def scanpy_int64_safe_pair_sampling():
    _scanpy_pp_utils.sample_comb = (
        sample_comb_int64_safe
    )
    _scrublet_core.sample_comb = (
        sample_comb_int64_safe
    )

    try:
        yield
    finally:
        _scanpy_pp_utils.sample_comb = (
            _ORIGINAL_UTILS_SAMPLE_COMB
        )
        _scrublet_core.sample_comb = (
            _ORIGINAL_CORE_SAMPLE_COMB
        )


def add_supported_seed_parameter(
    requested: dict,
    function,
    function_label: str,
) -> tuple[dict, str]:
    signature = inspect.signature(function)
    supported = set(signature.parameters)

    for keyword in (
        "random_seed",
        "random_state",
        "rng",
    ):
        if keyword in supported:
            requested[keyword] = RANDOM_SEED
            return requested, keyword

    raise RuntimeError(
        f"{function_label} exposes none of "
        "random_seed, random_state, or rng. "
        f"Observed signature: {signature}"
    )


def simulation_parameters() -> dict:
    requested = {
        "sim_doublet_ratio": SIM_DOUBLET_RATIO,
        "synthetic_doublet_umi_subsampling": 1.0,
    }

    signature = inspect.signature(
        sc.pp.scrublet_simulate_doublets
    )
    supported = set(signature.parameters)

    if "layer" in supported:
        requested["layer"] = None

    requested, _ = add_supported_seed_parameter(
        requested,
        sc.pp.scrublet_simulate_doublets,
        "scanpy.pp.scrublet_simulate_doublets",
    )

    missing = sorted(
        set(requested) - supported
    )
    if missing:
        raise RuntimeError(
            "Unsupported doublet-simulation "
            f"parameters: {missing}. "
            f"Observed signature: {signature}"
        )

    return requested


def core_scrublet_parameters(
    n_prin_comps: int,
) -> dict:
    requested = {
        "expected_doublet_rate": (
            EXPECTED_DOUBLET_RATE
        ),
        "n_prin_comps": int(n_prin_comps),
        "mean_center": False,
        "normalize_variance": True,
        "log_transform": False,
        "use_approx_neighbors": True,
        "copy": False,
    }

    requested, _ = add_supported_seed_parameter(
        requested,
        sc.pp.scrublet,
        "scanpy.pp.scrublet",
    )

    signature = inspect.signature(
        sc.pp.scrublet
    )
    supported = set(signature.parameters)
    missing = sorted(
        set(requested) - supported
    )

    if missing:
        raise RuntimeError(
            "Unsupported Scrublet core parameters: "
            f"{missing}. Observed signature: "
            f"{signature}"
        )

    return requested


In [4]:
print("Opening DS4 H5AD with direct HDF5 access")

with h5py.File(INPUT_H5AD, "r") as input_h5:
    if "X" not in input_h5:
        raise KeyError("The input H5AD is missing /X.")

    if (
        "layers" not in input_h5
        or "counts" not in input_h5["layers"]
    ):
        raise KeyError(
            "The post-QC object is missing "
            "/layers/counts."
        )

    if "obs" not in input_h5 or "var" not in input_h5:
        raise KeyError(
            "The input H5AD is missing /obs or /var."
        )

    observed_shape = stored_matrix_shape(
        input_h5["X"]
    )
    counts_group = input_h5["layers"]["counts"]
    counts_shape = stored_matrix_shape(
        counts_group
    )

    if observed_shape != EXPECTED_INPUT_SHAPE:
        raise AssertionError(
            f"Expected input shape "
            f"{EXPECTED_INPUT_SHAPE}, "
            f"observed {observed_shape}."
        )

    if counts_shape != EXPECTED_INPUT_SHAPE:
        raise AssertionError(
            "The counts layer has an unexpected "
            f"shape: {counts_shape}."
        )

    obs_output = read_elem(input_h5["obs"])
    var_output = read_elem(input_h5["var"])
    uns_output = (
        deepcopy(read_elem(input_h5["uns"]))
        if "uns" in input_h5
        else {}
    )

    raw_counts = load_full_csr(counts_group)

if len(obs_output) != EXPECTED_INPUT_SHAPE[0]:
    raise AssertionError(
        "The /obs table has an unexpected row count."
    )

if len(var_output) != EXPECTED_INPUT_SHAPE[1]:
    raise AssertionError(
        "The /var table has an unexpected row count."
    )

if "sample_id" not in obs_output.columns:
    raise KeyError("obs['sample_id'] is required.")

if not obs_output.index.is_unique:
    raise AssertionError(
        "Cell identifiers are not unique."
    )

if not var_output.index.is_unique:
    raise AssertionError(
        "Gene identifiers are not unique."
    )

for column in (
    "doublet_score",
    "predicted_doublet",
):
    if column in obs_output.columns:
        del obs_output[column]

uns_output.pop("scrublet", None)
uns_output.pop(
    "doublet_detection_provenance",
    None,
)

assert_raw_counts_chunked(
    raw_counts,
    "layers['counts']",
)

obs_names = pd.Index(
    obs_output.index.astype(str),
    dtype="object",
)
var_names = pd.Index(
    var_output.index.astype(str),
    dtype="object",
)

print("Input shape:", raw_counts.shape)
print("Counts nonzero entries:", f"{raw_counts.nnz:,}")
print(
    "Counts memory:",
    f"{(raw_counts.data.nbytes + raw_counts.indices.nbytes + raw_counts.indptr.nbytes) / 1024**3:.2f} GiB",
)


Opening DS4 H5AD with direct HDF5 access
Reading counts.data: 182,316,786 entries
Reading counts.indices
Reading counts.indptr
Input shape: (88094, 24584)
Counts nonzero entries: 182,316,786
Counts memory: 1.36 GiB


In [5]:
print("Selecting default Scanpy HVGs before Scrublet")

hvg_result = select_scanpy_default_hvgs(
    raw_counts,
    var_names,
)

cell_keep = hvg_result["cell_keep"]

if not cell_keep.all():
    raise AssertionError(
        "DS4 post-QC contains cells with fewer than "
        f"{SCRUBLET_MIN_GENES_PER_CELL} detected genes. "
        "The historical complete-dataset diagnostic "
        "cannot be mapped one-to-one without an explicit "
        "not-checked state."
    )

hvg_counts = hvg_result["hvg_counts"]
hvg_var_names = hvg_result["hvg_var_names"]

del raw_counts
gc.collect()

n_pcs = min(
    N_PCS,
    hvg_counts.shape[0] - 1,
    hvg_counts.shape[1] - 1,
)

if n_pcs < 2:
    raise RuntimeError(
        "Too few cells or HVGs for Scrublet."
    )

print(
    "Genes after min_cells=3:",
    hvg_result["genes_after_min_cells"],
)
print(
    "Highly variable genes:",
    hvg_result["highly_variable_genes"],
)
print("Scrublet principal components:", n_pcs)
print(
    "HVG count-matrix nonzeros:",
    f"{hvg_counts.nnz:,}",
)


Selecting default Scanpy HVGs before Scrublet
Creating the one temporary normalized matrix for default HVG selection
Genes after min_cells=3: 24584
Highly variable genes: 1955
Scrublet principal components: 30
HVG count-matrix nonzeros: 25,405,696


In [6]:
observed = ad.AnnData(
    X=hvg_counts,
    obs=pd.DataFrame(
        index=obs_names.copy()
    ),
    var=pd.DataFrame(
        index=hvg_var_names.copy()
    ),
)

simulated = None
simulation_kwargs = simulation_parameters()
scrublet_parameters = core_scrublet_parameters(
    n_pcs
)

pair_sampling_population = int(
    observed.n_obs * observed.n_obs
)

print("Simulation parameters:", simulation_kwargs)
print("Scrublet core parameters:", scrublet_parameters)
print(
    "Simulated-pair population:",
    f"{pair_sampling_population:,}",
)
print("Signed int32 limit:", f"{_INT32_MAX:,}")


try:
    with scanpy_int64_safe_pair_sampling():
        simulated = (
            sc.pp.scrublet_simulate_doublets(
                observed,
                **simulation_kwargs,
            )
        )

    sc.pp.normalize_total(
        observed,
        target_sum=1e6,
    )
    sc.pp.normalize_total(
        simulated,
        target_sum=1e6,
    )

    sc.pp.scrublet(
        observed,
        adata_sim=simulated,
        **scrublet_parameters,
    )

    required_columns = {
        "doublet_score",
        "predicted_doublet",
    }
    missing = (
        required_columns
        - set(observed.obs.columns)
    )

    if missing:
        raise RuntimeError(
            "Scrublet did not create required "
            f"columns: {missing}."
        )

    doublet_scores = pd.to_numeric(
        observed.obs["doublet_score"],
        errors="raise",
    ).to_numpy(dtype=np.float64)

    predicted_doublets = (
        observed.obs["predicted_doublet"]
        .astype(bool)
        .to_numpy()
    )

    if not np.isfinite(doublet_scores).all():
        raise ValueError(
            "Scrublet returned non-finite scores."
        )

    scrublet_uns = (
        deepcopy(observed.uns["scrublet"])
        if "scrublet" in observed.uns
        else {}
    )

finally:
    if simulated is not None:
        del simulated

    del observed
    del hvg_counts
    gc.collect()


obs_output["doublet_score"] = doublet_scores
obs_output["predicted_doublet"] = (
    predicted_doublets
)

n_doublets = int(
    predicted_doublets.sum()
)
n_singlets = int(
    (~predicted_doublets).sum()
)

historical_summary_match = (
    n_doublets
    == EXPECTED_PREDICTED_DOUBLETS
    and n_singlets
    == EXPECTED_PREDICTED_SINGLETS
)

print(
    obs_output["predicted_doublet"]
    .value_counts()
)
print(
    obs_output["doublet_score"]
    .describe()
)

if historical_summary_match:
    print(
        "Historical summary reproduced exactly:",
        f"{n_doublets} doublets and "
        f"{n_singlets} singlets.",
    )
else:
    print(
        "WARNING: the memory-safe sparse-SVD run "
        "differs from the historical centered-PCA "
        "summary. Observed:",
        f"{n_doublets} doublets and "
        f"{n_singlets} singlets.",
    )


Simulation parameters: {'sim_doublet_ratio': 2.0, 'synthetic_doublet_umi_subsampling': 1.0, 'layer': None, 'random_seed': 0}
Scrublet core parameters: {'expected_doublet_rate': 0.05, 'n_prin_comps': 30, 'mean_center': False, 'normalize_variance': True, 'log_transform': False, 'use_approx_neighbors': True, 'copy': False, 'random_state': 0}
Simulated-pair population: 7,760,552,836
Signed int32 limit: 2,147,483,647
predicted_doublet
False    87694
True       400
Name: count, dtype: int64
count    88094.000000
mean         0.031514
std          0.044343
min          0.001560
25%          0.012773
50%          0.022680
75%          0.037433
max          0.853846
Name: doublet_score, dtype: float64


In [7]:
per_sample_summary = (
    obs_output.groupby(
        "sample_id",
        observed=True,
    )
    .agg(
        cells=("predicted_doublet", "size"),
        predicted_doublets=(
            "predicted_doublet",
            "sum",
        ),
        mean_doublet_score=(
            "doublet_score",
            "mean",
        ),
        median_doublet_score=(
            "doublet_score",
            "median",
        ),
        maximum_doublet_score=(
            "doublet_score",
            "max",
        ),
    )
    .reset_index()
)

per_sample_summary[
    "predicted_doublet_fraction"
] = (
    per_sample_summary["predicted_doublets"]
    / per_sample_summary["cells"]
)

per_sample_summary.to_csv(
    AUDIT_DIR
    / "GSE278450_scrublet_summary_by_sample.tsv",
    sep="\t",
    index=False,
)

obs_output[
    [
        "sample_id",
        "doublet_score",
        "predicted_doublet",
    ]
].to_csv(
    AUDIT_DIR
    / "GSE278450_scrublet_per_cell_scores.tsv.gz",
    sep="\t",
    compression="gzip",
)

per_sample_summary.sort_values(
    "predicted_doublet_fraction",
    ascending=False,
)


,sample_id,cells,predicted_doublets,mean_doublet_score,median_doublet_score,maximum_doublet_score,predicted_doublet_fraction
18,GSM8546851_BT409,2117,28,0.047301,0.033054,0.853846,0.013226
17,GSM8546850_BT408,2836,26,0.025466,0.013689,0.853846,0.009168
14,GSM8546847_BT400,4911,42,0.024465,0.012267,0.795332,0.008552
4,GSM8546837_BT369,2617,20,0.029231,0.015074,0.744108,0.007642
5,GSM8546838_BT373,6241,46,0.024044,0.014095,0.795332,0.007371
1,GSM8546834_BT363,8928,63,0.025248,0.015950,0.744108,0.007056
9,GSM8546842_BT389A,4478,31,0.042899,0.034243,0.795332,0.006923
11,GSM8546844_BT390,3336,20,0.038831,0.030273,0.853846,0.005995
16,GSM8546849_BT407,3474,20,0.030105,0.023907,0.795332,0.005757
21,GSM8546854_BT426,3316,15,0.032711,0.024760,0.853846,0.004524


In [8]:
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.hist(
    obs_output["doublet_score"].to_numpy(),
    bins=80,
    edgecolor="black",
)
ax.set_xlabel("Scrublet doublet score")
ax.set_ylabel("Cells")
ax.set_title(
    "GSE278450 post-QC Scrublet scores"
)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR
    / "GSE278450_scrublet_score_histogram.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)


In [9]:
doublet_summary = {
    "dataset_id": "DS4_GSE278450",
    "geo_accession": "GSE278450",
    "scope": "complete post-QC DS4 object",
    "method": "scanpy.pp.scrublet",
    "input_cells": int(len(obs_output)),
    "input_genes": int(len(var_output)),
    "genes_after_min_cells": int(
        hvg_result["genes_after_min_cells"]
    ),
    "highly_variable_genes": int(
        hvg_result["highly_variable_genes"]
    ),
    "normalization_target_sum_for_hvg": float(
        hvg_result["normalization_target_sum"]
    ),
    "simulation_parameters": simulation_kwargs,
    "scrublet_core_parameters": (
        scrublet_parameters
    ),
    "pair_sampling_population": (
        pair_sampling_population
    ),
    "pair_sampling_int32_limit": _INT32_MAX,
    "predicted_doublets": n_doublets,
    "predicted_singlets": n_singlets,
    "predicted_doublet_fraction": float(
        n_doublets / len(obs_output)
    ),
    "historical_expected_doublets": (
        EXPECTED_PREDICTED_DOUBLETS
    ),
    "historical_expected_singlets": (
        EXPECTED_PREDICTED_SINGLETS
    ),
    "historical_summary_match": (
        historical_summary_match
    ),
    "doublets_removed_from_main_analysis": False,
    "main_analysis_cells": int(
        len(obs_output)
    ),
    "runtime_patch": (
        "direct HDF5 CSR read; pre-HVG filtering; "
        "int64-safe simulated-pair sampling; "
        "mean_center=False sparse TruncatedSVD; "
        "metadata-only H5AD patch"
    ),
    "runtime_patch_version": (
        RUNTIME_PATCH_VERSION
    ),
    "timestamp": datetime.now().isoformat(
        timespec="seconds"
    ),
    "output_h5ad": str(OUTPUT_H5AD),
}

if scrublet_uns:
    uns_output["scrublet"] = scrublet_uns

uns_output[
    "doublet_detection_provenance"
] = {
    "scope": (
        "post_QC_dataset_level_diagnostic"
    ),
    "method": "scanpy.pp.scrublet",
    "doublets_removed_from_main_analysis": (
        "false"
    ),
    "predicted_doublets": n_doublets,
    "predicted_singlets": n_singlets,
    "historical_summary_match": str(
        historical_summary_match
    ).lower(),
    "parameters_json": json.dumps(
        {
            "simulation": simulation_kwargs,
            "core": scrublet_parameters,
        },
        sort_keys=True,
    ),
    "runtime_patch": (
        doublet_summary["runtime_patch"]
    ),
    "runtime_patch_version": (
        RUNTIME_PATCH_VERSION
    ),
}

with open(
    AUDIT_DIR
    / "GSE278450_scrublet_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        doublet_summary,
        handle,
        indent=2,
    )

print(doublet_summary)


{'dataset_id': 'DS4_GSE278450', 'geo_accession': 'GSE278450', 'scope': 'complete post-QC DS4 object', 'method': 'scanpy.pp.scrublet', 'input_cells': 88094, 'input_genes': 24584, 'genes_after_min_cells': 24584, 'highly_variable_genes': 1955, 'normalization_target_sum_for_hvg': 5044.0, 'simulation_parameters': {'sim_doublet_ratio': 2.0, 'synthetic_doublet_umi_subsampling': 1.0, 'layer': None, 'random_seed': 0}, 'scrublet_core_parameters': {'expected_doublet_rate': 0.05, 'n_prin_comps': 30, 'mean_center': False, 'normalize_variance': True, 'log_transform': False, 'use_approx_neighbors': True, 'copy': False, 'random_state': 0}, 'pair_sampling_population': 7760552836, 'pair_sampling_int32_limit': 2147483647, 'predicted_doublets': 400, 'predicted_singlets': 87694, 'predicted_doublet_fraction': 0.004540604354439576, 'historical_expected_doublets': 310, 'historical_expected_singlets': 87784, 'historical_summary_match': False, 'doublets_removed_from_main_analysis': False, 'main_analysis_cells':

In [ ]:
if (
    len(obs_output),
    len(var_output),
) != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        "The doublet diagnostic unexpectedly "
        "changed the object shape."
    )

temporary_output = OUTPUT_H5AD.with_name(
    f"{OUTPUT_H5AD.stem}.temporary"
    f"{OUTPUT_H5AD.suffix}"
)
metadata_sidecar = OUTPUT_H5AD.with_name(
    f"{OUTPUT_H5AD.stem}.metadata_sidecar"
    f"{OUTPUT_H5AD.suffix}"
)

for path in (
    temporary_output,
    metadata_sidecar,
):
    if path.exists():
        path.unlink()

input_size = INPUT_H5AD.stat().st_size
free_space = shutil.disk_usage(
    OUTPUT_H5AD.parent
).free
required_free_space = (
    input_size + 2 * 1024**3
)

if free_space < required_free_space:
    raise OSError(
        "Insufficient free disk space for the "
        "atomic output copy. "
        f"Input size: "
        f"{input_size / 1024**3:.2f} GiB; "
        f"free space: "
        f"{free_space / 1024**3:.2f} GiB."
    )

print("Copying the large input H5AD on disk...")

shutil.copy2(
    INPUT_H5AD,
    temporary_output,
)

metadata_holder = ad.AnnData(
    X=sp.csr_matrix(
        (len(obs_output), 0),
        dtype=np.float32,
    ),
    obs=obs_output,
    var=pd.DataFrame(
        index=pd.Index([], dtype=str)
    ),
    uns=uns_output,
)

metadata_holder.write_h5ad(
    metadata_sidecar,
    compression="lzf",
)

del metadata_holder
gc.collect()


try:
    with h5py.File(
        metadata_sidecar,
        "r",
    ) as source_handle:
        with h5py.File(
            temporary_output,
            "r+",
        ) as output_handle:
            for group_name in ("obs", "uns"):
                if group_name not in source_handle:
                    raise AssertionError(
                        "Metadata sidecar is missing "
                        f"/{group_name}."
                    )

                if group_name in output_handle:
                    del output_handle[group_name]

                source_handle.copy(
                    group_name,
                    output_handle,
                    name=group_name,
                )

            output_handle.flush()

    with h5py.File(
        temporary_output,
        "r",
    ) as check_handle:
        if "X" not in check_handle:
            raise AssertionError(
                "Saved output is missing /X."
            )

        if (
            stored_matrix_shape(
                check_handle["X"]
            )
            != EXPECTED_INPUT_SHAPE
        ):
            raise AssertionError(
                "Saved X has an unexpected shape."
            )

        if (
            "layers" not in check_handle
            or "counts"
            not in check_handle["layers"]
        ):
            raise AssertionError(
                "Saved output is missing "
                "layers['counts']."
            )

        if (
            stored_matrix_shape(
                check_handle["layers"]["counts"]
            )
            != EXPECTED_INPUT_SHAPE
        ):
            raise AssertionError(
                "Saved counts layer has an "
                "unexpected shape."
            )

        saved_obs = read_elem(
            check_handle["obs"]
        )
        saved_uns = read_elem(
            check_handle["uns"]
        )

    for column in (
        "doublet_score",
        "predicted_doublet",
    ):
        if column not in saved_obs.columns:
            raise AssertionError(
                "Saved output is missing "
                f"obs[{column!r}]."
            )

    if (
        "doublet_detection_provenance"
        not in saved_uns
    ):
        raise AssertionError(
            "Saved output is missing "
            "uns['doublet_detection_provenance']."
        )

    if int(
        saved_obs["predicted_doublet"].sum()
    ) != n_doublets:
        raise AssertionError(
            "Saved predicted-doublet count "
            "does not match the run result."
        )

    temporary_output.replace(
        OUTPUT_H5AD
    )

finally:
    if metadata_sidecar.exists():
        metadata_sidecar.unlink()


print()
print("Saved and validated:", OUTPUT_H5AD)
print("Validated shape:", EXPECTED_INPUT_SHAPE)
print("Validated layer: counts")
print(
    "Validated obs columns: "
    "doublet_score, predicted_doublet"
)
print(
    "Validated uns entry: "
    "doublet_detection_provenance"
)
print(
    "Historical summary match:",
    historical_summary_match,
)


## Reproducibility note

The DS4 Scrublet analysis was retained as a diagnostic step only, and
predicted doublets were not excluded from downstream analyses.

The historical run reported 310 predicted doublets and 87,784 predicted
singlets. On the current Windows system with 16 GB RAM, the original
mean-centered PCA implementation exceeded the available memory.
Therefore, the reproducibility run used `mean_center=False`, preserving
sparse matrices and using TruncatedSVD.

Consequently, the current Scrublet calls were not expected to reproduce
the historical cell-level calls exactly. This difference does not alter
the expression matrix, QC-filtered cell set, CNV analysis, clustering, or
other downstream results because no cells were removed based on the
Scrublet diagnostic.